In [ ]:
import os, re, ast, warnings, time, random
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity   # kept for eval only
import chromadb
from chromadb import Settings
from sentence_transformers import SentenceTransformer
warnings.filterwarnings("ignore")
matplotlib.use("Agg")
sns.set_theme(style="darkgrid", palette="muted")
pd.set_option("display.max_colwidth", 80)
print("All imports done")

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

def _make_driver(headless=True):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    # Selenium 4.6+ manages chromedriver automatically — no Service/path needed
    return webdriver.Chrome(options=opts)

def _parse_runtime(text):
    if not text:
        return None
    m = re.search(r"(\d+)", str(text).replace(" min",""))
    return int(m.group(1)) if m else None

def scrape_imdb_top(target=500, headless=True):
    url = "https://www.imdb.com/search/title/?groups=top_1000&sort=user_rating,desc&count=50"
    driver = _make_driver(headless)
    rows = []
    try:
        driver.get(url)
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item")
            )
        )
        for card in driver.find_elements(By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item")[:target]:
            try:
                title = card.find_element(By.CSS_SELECTOR, "a.ipc-title-link-wrapper").text
                title = re.sub(r"^\s*\d+\.\s*", "", title).strip()
                rows.append({"title": title})
            except Exception:
                continue
    finally:
        driver.quit()
    print(f"Scraped {len(rows)} items.")
    return pd.DataFrame(rows)

print("Scraper defined.")

In [ ]:
# Scraping Demo + Load 10000 movie IMDB dataset

SCRAPE_CACHE = "NLP/MovieMate_NLP_Assignment-1/imdb_top500_selenium.csv"

if os.path.exists(SCRAPE_CACHE):
    df_scraped = pd.read_csv(SCRAPE_CACHE)
    print(f"Loaded scraped data from cache: {df_scraped.shape}")
elif 'scrape_imdb_top' in dir():
    print("Running Selenium scraper")
    df_scraped = scrape_imdb_top(target=500, headless=True)
    df_scraped.to_csv(SCRAPE_CACHE, index=False)
    print(f"Scraped and saved: {df_scraped.shape}")
else:
    df_scraped = pd.DataFrame(columns=["title"])

if not df_scraped.empty:
    print(f"Scraped columns : {list(df_scraped.columns)}")
    print(df_scraped.head(3))

print("\n" + "="*60)

# Main dataset:Kaggle 10K IMDB
DATA_PATH = "NLP/MovieMate_NLP_Assignment-1/Top_10000_Movies_IMDb.csv"
df_raw = pd.read_csv(DATA_PATH)
print("RAW DATASET (Kaggle 10K)")
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print()
print("Missing values:")
print(df_raw.isnull().sum())
print()
df_raw.head(3)
# chrome driver needed for scraping top 500 movies.

In [ ]:
# Data Preprocessing
import ast

df = df_raw.rename(columns={
    "Movie Name": "title",
    "Rating":     "rating",
    "Runtime":    "duration_min",
    "Genre":      "genres",
    "Metascore":  "metascore",
    "Plot":       "plot",
    "Directors":  "_directors_raw",
    "Stars":      "_stars_raw",
    "Votes":      "vote_count",
    "Gross":      "gross",
    "Link":       "url",
}).copy()

# Parse director and cast from list-strings
def _parse_list(val):
    try: return ast.literal_eval(str(val))
    except: return []

df["_dir_list"]  = df["_directors_raw"].apply(_parse_list)
df["_star_list"] = df["_stars_raw"].apply(_parse_list)
star_set = df["_star_list"].apply(set)
df["director"] = df.apply(
    lambda r: ",".join([x for x in r["_dir_list"] if x not in star_set[r.name]]) or "", axis=1)
df["cast"] = df["_star_list"].apply(lambda x: ",".join(x[:4]))
df.drop(columns=["_directors_raw","_stars_raw","_dir_list","_star_list"], inplace=True)

# Numeric coercion
df["rating"]       = pd.to_numeric(df["rating"],    errors="coerce")
df["duration_min"] = df["duration_min"].astype(str).str.replace(r"[^\d]","",regex=True)
df["duration_min"] = pd.to_numeric(df["duration_min"], errors="coerce").astype("Int64")
df["vote_count"]   = pd.to_numeric(df["vote_count"],   errors="coerce").fillna(0).astype(int)
df["metascore"]    = pd.to_numeric(df["metascore"],    errors="coerce")

# Text cleanup
for col in ["title","genres","plot","director","cast"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

df = df.drop_duplicates(subset="title").reset_index(drop=True)
df["primary_genre"] = df["genres"].str.split(",").str[0].str.strip()
df["rating_label"]  = pd.cut(
    df["rating"],
    bins=[0, 7.0, 7.9, 8.4, 10],
    labels=["Good (7+)","Very Good (7-8)","Great (8-8.4)","Masterpiece (8.5+)"]
)

print(f"Preprocessing complete. Dataset: {df.shape}")
df[["title","rating","duration_min","genres","director"]].head(5)

In [ ]:
# Rating Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["rating"].dropna(), bins=30, color="steelblue", edgecolor="white", lw=0.5)
axes[0].set_title("IMDb Rating Distribution", fontsize=13)
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")
label_counts = df["rating_label"].value_counts()
axes[1].bar(label_counts.index, label_counts.values,color=sns.color_palette("muted", len(label_counts)))
axes[1].set_title("Movies by Rating Category", fontsize=13)
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Count")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("plot_ratings.png", dpi=120, bbox_inches="tight")
plt.show()
print("Rating stats:", df["rating"].describe().round(2).to_dict())

In [ ]:
# Genre Frequency
genre_series = (
    df["genres"]
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
    .head(15)
)
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=genre_series.values, y=genre_series.index, palette="viridis", ax=ax)
ax.set_title("Top 15 Genres", fontsize=13)
ax.set_xlabel("Number of Movies")
plt.tight_layout()
plt.savefig("plot_genres.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Top Directors by Film Count
top_dirs = (
    df["director"]
    .str.split(",").str[0].str.strip()
    .replace("", pd.NA).dropna()
    .value_counts()
    .head(12)
)
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=top_dirs.values, y=top_dirs.index, palette="rocket", ax=ax)
ax.set_title("Top 12 Directors by Film Count", fontsize=13)
ax.set_xlabel("Number of Movies")
plt.tight_layout()
plt.savefig("plot_directors.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Average Rating by Primary Genre
genre_rating = (
    df.groupby("primary_genre")["rating"]
    .agg(["mean","count"])
    .query("count >= 10")
    .sort_values("mean", ascending=False)
    .head(15)
)
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=genre_rating["mean"], y=genre_rating.index, palette="coolwarm", ax=ax)
ax.set_title("Average Rating by Genre", fontsize=13)
ax.set_xlabel("Avg IMDb Rating")
ax.axvline(df["rating"].mean(), color="red", ls="--", label="Overall mean")
ax.legend()
plt.tight_layout()
plt.savefig("plot_genre_rating.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Rating vs Votes
sample = df[df["vote_count"] > 1000].sample(min(2000, len(df)), random_state=42)
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(
    sample["vote_count"], sample["rating"],
    alpha=0.3, s=15, c=sample["rating"], cmap="RdYlGn"
)
ax.set_xscale("log")
ax.set_title("Rating vs Vote Count (log scale)", fontsize=13)
ax.set_xlabel("Votes (log)")
ax.set_ylabel("IMDb Rating")
plt.tight_layout()
plt.savefig("plot_rating_votes.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
#  5.  Build Enriched Text Corpus for Embedding
def build_movie_text(row: pd.Series) -> str:
    parts = []
    if row["title"]:
        parts.append(row["title"] + ".")
    if row["genres"]:
        parts.append(f"Genre: {row['genres']}.")
    if row["director"]:
        parts.append(f"Director: {row['director']}.")
    if row["cast"]:
        parts.append(f"Stars: {row['cast']}.")
    if row["plot"]:
        parts.append(row["plot"])
    return " ".join(parts)

df["corpus_text"] = df.apply(build_movie_text, axis=1)

print("Sample corpus text:")
print(df["corpus_text"].iloc[0])
print()
print(f"Corpus built for {len(df)} movies.")

In [ ]:
# Sentence Transformer Embeddings + ChromaDB Vector Store
EMBED_MODEL = "all-MiniLM-L6-v2"   # 384-dim
CHROMA_PATH = "../re/chroma_moviemate"

print(f"Loading embedding model: {EMBED_MODEL}...")
embedder = SentenceTransformer(EMBED_MODEL)
print("Model loaded.")

# Generate embeddings (batched)
print(f"Encoding corpus for {len(df)} movies...")
corpus_texts = df["corpus_text"].tolist()
embeddings   = embedder.encode(
    corpus_texts,
    batch_size     = 128,
    show_progress_bar = True,
    normalize_embeddings = True,   # cosine via dot product
)
print(f"Embeddings shape:{embeddings.shape}")

#  Build ChromaDB collection
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    chroma_client.delete_collection("movies")
except Exception:
    pass

collection = chroma_client.create_collection(
    name     = "movies",
    metadata = {"hnsw:space": "cosine"},
)

# Add in batches of 500
BATCH = 500
for start in range(0, len(df), BATCH):
    end = min(start + BATCH, len(df))
    batch_df = df.iloc[start:end]
    collection.add(
        ids = [str(i) for i in range(start, end)],
        embeddings = embeddings[start:end].tolist(),
        documents  = corpus_texts[start:end],
        metadatas  = [
            {
                "title"        : row["title"],
                "rating"       : float(row["rating"]) if pd.notna(row["rating"]) else 0.0,
                "genres"       : row["genres"],
                "director"     : row["director"],
                "cast"         : row["cast"],
                "duration_min" : int(row["duration_min"]) if pd.notna(row["duration_min"]) else 0,
                "vote_count"   : int(row["vote_count"]),
                "plot"         : row["plot"][:300],
                "url"          : row["url"],
                "df_index"     : int(start + (batch_df.index.tolist().index(batch_df.index[end-start-1-(end-start-1)])
                                              if False else i - start)),
            }
            for i, (_, row) in enumerate(batch_df.iterrows())
        ],
    )
    print(f"Indexed {end}/{len(df)} movies")

print(f"\nChromaDB collection ready:{collection.count()} vectors in '{CHROMA_PATH}'")

In [ ]:
# Vector Search Function (RAG Retrieval)
def vector_search(
    query : str,
    top_k : int   = 5,
    min_rating : float = 0.0,
    max_duration : int   = 9999,
    genre_filter : str   = None,
    director_filter: str = None,
    candidate_k  : int   = 50,
) -> pd.DataFrame:
    query_vec = embedder.encode([query], normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings = query_vec,
        n_results        = min(candidate_k, collection.count()),
        include          = ["metadatas", "distances", "documents"],
    )
    metas     = results["metadatas"][0]
    distances = results["distances"][0]

    if not metas:
        return pd.DataFrame()

    hits = pd.DataFrame(metas)
    hits["score"] = [1 - d for d in distances]   # cosine similarity

    #  Post-filter
    if min_rating > 0:
        hits = hits[hits["rating"] >= min_rating]
    if max_duration < 9999:
        hits = hits[hits["duration_min"].between(1, max_duration)]
    if genre_filter:
        hits = hits[hits["genres"].str.contains(genre_filter, case=False, na=False)]
    if director_filter:
        hits = hits[hits["director"].str.contains(director_filter, case=False, na=False)]

    return hits.head(top_k).reset_index(drop=True)


In [ ]:
# Intent / Constraint Parser
GENRE_MAP = {
    "sci-fi":"Sci-Fi","scifi":"Sci-Fi","science fiction":"Sci-Fi",
    "action":"Action","adventure":"Adventure","thriller":"Thriller",
    "comedy":"Comedy","horror":"Horror","romance":"Romance",
    "drama":"Drama","animation":"Animation","animated":"Animation",
    "documentary":"Documentary","fantasy":"Fantasy","mystery":"Mystery",
    "crime":"Crime","war":"War","biography":"Biography","history":"History",
    "family":"Family","music":"Music","sport":"Sport","western":"Western",
}

def parse_intent(query: str) -> dict:
    q = query.lower()

    # Rating
    min_rating = 0.0
    m = re.search(r"rating[\s>]+([\d.]+)", q) or re.search(r"above\s+([\d.]+)", q)
    if m:
        min_rating = float(m.group(1))
    elif any(w in q for w in ["masterpiece","greatest","best"]):
        min_rating = 8.5
    elif any(w in q for w in ["highly rated","top rated","excellent"]):
        min_rating = 8.0

    # Duration
    max_dur = 9999
    m2 = re.search(r"(under|less than|max|within)\s+(\d+)\s*(min|hour|hr)", q)
    if m2:
        val = int(m2.group(2))
        max_dur = val if "min" in m2.group(3) else val * 60

    # Genre
    genre = None
    for alias, canonical in GENRE_MAP.items():
        if alias in q:
            genre = canonical
            break

    # Director
    director = None
    m3 = re.search(
        r"(?:by|directed by|from director)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})",
        query,
    )
    if m3:
        director = m3.group(1)

    # Similarity query  ("similar to X" / "like X")
    sim_title = None
    m4 = re.search(r'(?:similar to|like)\s+["\']?([\w\s]+?)["\']?(?:\s|$)', q)
    if m4:
        sim_title = m4.group(1).strip().title()

    semantic_query = sim_title or query

    return {
        "semantic_query": semantic_query,
        "min_rating":     min_rating,
        "max_duration":   max_dur,
        "genre_filter":   genre,
        "director_filter":director,
    }

print("Intent parser ready.")
print("Test:", parse_intent("Sci-fi movies with rating above 8.0 under 120 min"))
print("Test similar:", parse_intent("Similar to Inception"))

In [ ]:
# Formatting + LLM Response
def format_movie_card(row: pd.Series) -> str:
    dur  = f"{row.get('duration_min',0)} min" if row.get("duration_min",0) > 0 else "N/A"
    rat  = f"{row['rating']:.1f}" if row.get("rating") else "N/A"
    score = f" | Similarity: {row.get('score',0):.2f}" if "score" in row else ""
    return (
        f" {row['title']} ({rat}){score}\n"
        f"   Genre: {row.get('genres','N/A')}  |  Runtime: {dur}\n"
        f"   Director: {row.get('director','N/A')}\n"
        f"   Cast: {row.get('cast','N/A')}\n"
        f"   {row.get('plot','')[:140]}..."
    )

# Set LLM_PROVIDER = 'anthropic' or 'openai' if adventures 'grok' to use a real LLM
LLM_PROVIDER = "mock"

def _build_prompt(results: pd.DataFrame, query: str) -> str:
    ctx = "\n".join(
        f"- {r['title']} | {r.get('genres','')} | Rating: {r.get('rating','')} "
        f"| Director: {r.get('director','')}\n  Plot: {str(r.get('plot',''))[:100]}"
        for _, r in results.iterrows()
    )
    return (
        f"You are MovieMate, a knowledgeable movie recommendation assistant.\n"
        f"User asked: \"{query}\"\n\nRetrieved movies:\n{ctx}\n\n"
        f"Write a warm 2-3 sentence response explaining why these match, "
        f"then give a one-line description of each."
    )

def generate_response(results: pd.DataFrame, query: str) -> str:
    if results.empty:
        return "Sorry, I didnt find any movies matching. Try different keywords!"

    if LLM_PROVIDER == "mock":
        cards = "\n\n".join(format_movie_card(r) for _, r in results.iterrows())
        return f"Here are {len(results)} movies matching \"{query}\":\n\n{cards}"

    prompt = _build_prompt(results, query)

    if LLM_PROVIDER == "anthropic":
        import anthropic
        client = anthropic.Anthropic()
        msg = client.messages.create(
            model="claude-opus-4-6", max_tokens=512,
            messages=[{"role":"user","content":prompt}],
        )
        return msg.content[0].text

    if LLM_PROVIDER == "openai":
        import openai
        resp = openai.ChatCompletion.create(
            model="gpt-3.5-turbo", max_tokens=512,
            messages=[{"role":"user","content":prompt}],
        )
        return resp.choices[0].message.content

    return generate_response.__wrapped__(results, query)   # fallback mock

print("Response generator ready. Provider:", LLM_PROVIDER)

In [ ]:
# MovieMate Chatbot Class
HELP_TEXT = """
MovieMate  Conversational IMDB Movie Chatbot (RAG + ChromaDB)

You can ask me things like:
   "Suggest highly rated sci-fi movies"
   "Movies by Christopher Nolan"
   "Similar to Inception"
   "Best crime dramas with rating above 8.5"
   "Good thrillers under 2 hours"
   "Top action movies"
Type 'quit' to exit the chat.
"""

class MovieMateChatbot:

    def __init__(self, top_k=5):
        self.top_k         = top_k
        self.history       = []
        self._last_intent  = {}
        print(f"MovieMate ready  |  Dataset: {len(df)} movies  |  Provider: {LLM_PROVIDER}")

    def _merge_context(self, new_intent: dict) -> dict:
        if not self._last_intent:
            return new_intent
        merged = dict(self._last_intent)
        for key, val in new_intent.items():
            is_default = (
                (key == "min_rating"   and val == 0.0)  or
                (key == "max_duration" and val == 9999) or
                val is None
            )
            if not is_default:
                merged[key] = val
        merged["semantic_query"] = new_intent["semantic_query"] or self._last_intent.get("semantic_query","")
        return merged

    def chat(self, query: str) -> str:
        query = query.strip()
        if not query:
            return "Please type a movie query!"
        if query.lower() in ("help","?"):
            return HELP_TEXT

        intent  = parse_intent(query)
        intent  = self._merge_context(intent)
        results = vector_search(
            query          = intent["semantic_query"],
            top_k          = self.top_k,
            min_rating     = intent["min_rating"],
            max_duration   = intent["max_duration"],
            genre_filter   = intent["genre_filter"],
            director_filter= intent["director_filter"],
        )
        response = generate_response(results, query)
        self._last_intent = intent
        self.history.append({"user": query, "bot": response})
        return response
bot = MovieMateChatbot(top_k=5)

In [ ]:
print("Genre + Rating")
print(f"\nUser: Suggest highly rated sci-fi movies\n")
print(f"Bot:\n{bot.chat(repr('Suggest highly rated sci-fi movies')[1:-1])}")

In [ ]:
print("Director Query")
print(f"\nUser: Movies by Christopher Nolan\n")
print(f"Bot:\n{bot.chat(repr('Movies by Christopher Nolan')[1:-1])}")

In [ ]:
print("Similarity Search")
print(f"\nUser: Similar to Inception\n")
print(f"Bot:\n{bot.chat(repr('Similar to Inception')[1:-1])}")

In [ ]:
print("Complex Query")
print(f"\nUser: Best crime dramas with rating above 8.5\n")
print(f"Bot:\n{bot.chat(repr('Best crime dramas with rating above 8.5')[1:-1])}")

In [ ]:
print("Duration Filter")
print(f"\nUser: Good thrillers under 120 min\n")
print(f"Bot:\n{bot.chat(repr('Good thrillers under 120 min')[1:-1])}")

In [ ]:
print("Help")
print(f"\nUser: help\n")
print(f"Bot:\n{bot.chat(repr('help')[1:-1])}")

In [ ]:
#  Multi-Turn Conversation
print("Multi-Turn Conversation")
bot2  = MovieMateChatbot(top_k=5)
turns = [
    "Suggest action movies",
    "Only those with rating above 8",
    "Can you show thrillers instead",
]
for t in turns:
    print(f"User: {t}")
    print(f"Bot:\n{bot2.chat(t)}\n")

In [ ]:
# Evaluation  Precision @ K
EVAL_QUERIES = [
    {"query": "crime drama mafia family",           "expected": "The Godfather"},
    {"query": "dream heist mind-bending sci-fi",    "expected": "Inception"},
    {"query": "prison friendship hope redemption",  "expected": "The Shawshank Redemption"},
    {"query": "superhero dark knight Gotham",       "expected": "The Dark Knight"},
    {"query": "animated toys adventure friendship", "expected": "Toy Story"},
    {"query": "space war rebellion empire",         "expected": "Star Wars"},
    {"query": "ring fellowship fantasy quest",      "expected": "The Lord of the Rings"},
    {"query": "serial killer detective psychological thriller", "expected": "Se7en"},
]

def precision_at_k(queries, k=5):
    hits = 0
    for q in queries:
        results = vector_search(q["query"], top_k=k)
        if not results.empty and q["expected"].lower() in results["title"].str.lower().values:
            hits += 1
    p_at_k = hits / len(queries)
    return p_at_k, hits, len(queries)

p, h, total = precision_at_k(EVAL_QUERIES, k=5)
print(f"Precision@5 : {p:.2%}  ({h}/{total} queries hit)")

p10, h10, _ = precision_at_k(EVAL_QUERIES, k=10)
print(f"Precision@10: {p10:.2%}  ({h10}/{total} queries hit)")

In [ ]:
#  Similarity Score Distribution
sample_queries = [
    "crime drama", "sci-fi space adventure", "animated family fantasy",
    "psychological thriller mystery", "war history epic",
]
all_scores = []
for sq in sample_queries:
    res = vector_search(sq, top_k=20, candidate_k=50)
    all_scores.extend(res["score"].tolist())

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(all_scores, bins=25, color="darkorange", edgecolor="white", lw=0.5)
ax.set_title("Cosine Similarity Score Distribution (sample queries)", fontsize=13)
ax.set_xlabel("Cosine Similarity")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("plot_similarity_dist.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Mean similarity: {np.mean(all_scores):.3f}  |  Std: {np.std(all_scores):.3f}")

In [ ]:
# System Summary
p, h, total = precision_at_k(EVAL_QUERIES, k=5)
print("MOVIEMATE SYSTEM SUMMARY")
print(f"Dataset : {len(df):,} movies (IMDB Top 10 000)")
print(f"Embedding model : all-MiniLM-L6-v2 (384-dim)")
print(f"Vector store: ChromaDB ")
print(f"Retrieval : Semantic search + metadata post-filter")
print(f"LLM provider : {LLM_PROVIDER}")
print(f"Precision@5 : {p:.2%} ({h}/{total})")
print("\nEval query breakdown:")
for q in EVAL_QUERIES:
    res = vector_search(q["query"], top_k=5)
    found = q["expected"].lower() in res["title"].str.lower().values if not res.empty else False
    mark = "yes" if found else "no"
    print(f"  {mark}  [{q['expected']}]  query='{q['query']}'")

In [ ]:
#  Text box (terminal)
repl_bot = MovieMateChatbot(top_k=5)
print("MovieMate text chat. Type 'quit' to exit.  (max 20 turns)")

for _ in range(20):
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nSession ended.")
        break
    if not user_input or user_input.lower() in ("quit","exit","q"):
        print("Goodbye!")
        break
    print(f"\nBot:\n{repl_bot.chat(user_input)}\n")

In [ ]:
# Gradio Interactive Web Interface
import subprocess, sys
import gradio as gr

_ui_bot = MovieMateChatbot(top_k=5)

def chat_fn(user_message: str, history: list):
    if not user_message.strip():
        return history, history
    reply = _ui_bot.chat(user_message)
    history = history or []
    history.append({"role":"user","content": user_message})
    history.append({"role":"assistant","content": reply})
    return history, history

with gr.Blocks(title="MovieMate  IMDB AI Chatbot") as demo:
    gr.Markdown(
        "## MovieMate  IMDB AI Chatbot\n"
        "> **10 000 movies  Semantic search  ChromaDB  RAG pipeline**\n\n"
        "Ask anything: genre, director, mood, similar titles, rating filters"
    )
    chatbot = gr.Chatbot(label="MovieMate", height=450)
    state   = gr.State([])

    with gr.Row():
        txt = gr.Textbox(
            placeholder="e.g. Suggest sci-fi movies with rating above 8",
            show_label=False, scale=8,
        )
        btn = gr.Button("Send", variant="primary", scale=1)

    btn.click(chat_fn, inputs=[txt, state], outputs=[chatbot, state])
    txt.submit(chat_fn, inputs=[txt, state], outputs=[chatbot, state])

    gr.Examples(
        examples=[
            ["Suggest highly rated sci-fi movies"],
            ["Movies by Christopher Nolan"],
            ["Similar to Inception"],
            ["Best crime dramas with rating above 8.5"],
            ["Good thrillers under 120 minutes"],
            ["Top animated movies for kids"],
            ["War movies with rating above 8.0 directed by Steven Spielberg"],
        ],
        inputs=txt,
    )

demo.launch(share=True)